In [1]:
print("Geisha, Panama, Colombia, Costa Rica, Kerinci Sungai Penuh")

Geisha, Panama, Colombia, Costa Rica, Kerinci Sungai Penuh


In [ ]:
#  misal kedalaman sumur 100 m dengan diameter 25 cm
#  dengan kedalaman sensor dan pompa 55 m 
# dan tma mulai dari 0.0 hinggan 30.0 m [karena tidak boleh melebihi kedalaman sensor]
# buatkan data dummy dengan korelasi antara Data Air Tanah dan Muka Air Tanah, jadi Data Air Tanah merupakan selisih dari kedalaman sumur - Tinggi Muka Air Tanah

In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def generate_jiat_dummy_data(days=30, interval_minutes=60):
    """
    Generate synthetic hydrological data for EDA.
    Spek: Sumur 100m, Sensor 55m, TMA Range 0-30m.
    Timestamp ditetapkan sebagai Index DataFrame.
    """
    np.random.seed(42)
    
    # 1. Konstanta Spesifikasi Sumur
    KEDALAMAN_SUMUR = 100.0
    KEDALAMAN_SENSOR = 55.0
    DIAMETER = 0.25 
    
    # 2. Setup Index Waktu (DatetimeIndex)
    periods = int((days * 24 * 60) / interval_minutes)
    time_index = pd.date_range(start='2026-01-01', periods=periods, freq=f'{interval_minutes}min')
    
    tma_list = []
    current_tma = 1.5 
    
    # 3. Simulasi Hidrologi Harian
    for dt in time_index:
        hour = dt.hour
        discharge_on = 6 <= hour < 14
        
        if discharge_on:
            target_tma = 28.0 
            current_tma += (target_tma - current_tma) * 0.15 
        else:
            static_tma = 1.5
            current_tma += (static_tma - current_tma) * 0.10 
            
        noise = np.random.normal(0, 0.2)
        simulated_tma = current_tma + noise
        simulated_tma = max(0.0, min(30.0, simulated_tma))
        
        tma_list.append(simulated_tma)
        
    # LANGSUNG MENJADIKAN TIME_INDEX SEBAGAI INDEX DATAFRAME
    df = pd.DataFrame({
        'Muka_Air_Tanah': tma_list
    }, index=time_index)
    df.index.name = 'Timestamp' # Beri nama index untuk kejelasan
    
    # 4. KORELASI UTAMA
    df['Data_Air_Tanah'] = KEDALAMAN_SUMUR - df['Muka_Air_Tanah']
    
    # Tambahan fitur numerik
    df['Kedalaman_Sumur'] = KEDALAMAN_SUMUR
    df['Kedalaman_Sensor'] = KEDALAMAN_SENSOR
    df['Diameter_Sumur'] = DIAMETER
    
    # Mengakses jam langsung dari index via `df.index.hour`
    df['Discharge_Status'] = df.index.hour.map(lambda h: 1 if 6 <= h < 14 else 0)
    
    return df

# ==========================================
# EKSEKUSI & PENGUJIAN VISUAL EDA AWAL
# ==========================================
df_sumur = generate_jiat_dummy_data(days=120, interval_minutes=60) 

print("Data Sample Teratas (Perhatikan Timestamp kini adalah Index):")
display(df_sumur.head())

# VISUALISASI MENGGUNAKAN PLOTLY
fig = go.Figure()

# 1. Tambahkan Kurva Muka Air Tanah (TMA)
fig.add_trace(go.Scatter(
    x=df_sumur.index, # Menggunakan index
    y=df_sumur['Muka_Air_Tanah'],
    mode='lines',
    name='Muka Air Tanah (TMA)',
    line=dict(color='#ef4444', width=2.5)
))

# 2. Tambahkan Kurva Data Air Tanah (Sisa)
fig.add_trace(go.Scatter(
    x=df_sumur.index, # Menggunakan index
    y=df_sumur['Data_Air_Tanah'],
    mode='lines',
    name='Sisa Data Air Tanah',
    line=dict(color='#3b82f6', width=2.5)
))

# 3. Warnai Background Abu-abu pada Fase Discharge
discharge_data = df_sumur[df_sumur['Discharge_Status'] == 1]
# 'timestamp' di iterrows otomatis merepresentasikan nilai index
for timestamp, row in discharge_data.iterrows():
    fig.add_vrect(
        x0=timestamp, 
        x1=timestamp + pd.Timedelta(minutes=60),
        fillcolor="pink", 
        opacity=0.1,    
        layer="below",  
        line_width=0,
    )

# 4. Konfigurasi Kosmetik (Layout)
fig.update_layout(
    title='<b>Analisis Hubungan Sumur: Fluktuasi TMA vs Sisa Air (Fase Charge & Discharge)</b>',
    xaxis_title='Waktu Simulasi',
    yaxis_title='Kedalaman / Ketinggian (Meter)',
    hovermode='x unified', 
    template='plotly_dark',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

Data Sample Teratas (Perhatikan Timestamp kini adalah Index):


,Muka_Air_Tanah,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
Timestamp,,,,,,
2026-01-01 00:00:00,1.599343,98.400657,100.0,55.0,0.25,0
2026-01-01 01:00:00,1.472347,98.527653,100.0,55.0,0.25,0
2026-01-01 02:00:00,1.629538,98.370462,100.0,55.0,0.25,0
2026-01-01 03:00:00,1.804606,98.195394,100.0,55.0,0.25,0
2026-01-01 04:00:00,1.453169,98.546831,100.0,55.0,0.25,0


In [19]:
# df_sumur = df_sumur.reset_index()
 # df_sumur.to_csv
df_sumur.to_csv("data_tmat_dummy.csv", index=False)

In [9]:
df_sumur.loc["2024-01-01"]

,Muka_Air_Tanah,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
Timestamp,,,,,,
2024-01-01 00:00:00,1.599343,98.400657,100.0,55.0,0.25,0
2024-01-01 01:00:00,1.472347,98.527653,100.0,55.0,0.25,0
2024-01-01 02:00:00,1.629538,98.370462,100.0,55.0,0.25,0
2024-01-01 03:00:00,1.804606,98.195394,100.0,55.0,0.25,0
2024-01-01 04:00:00,1.453169,98.546831,100.0,55.0,0.25,0
2024-01-01 05:00:00,1.453173,98.546827,100.0,55.0,0.25,0
2024-01-01 06:00:00,5.790843,94.209157,100.0,55.0,0.25,1
2024-01-01 07:00:00,9.007237,90.992763,100.0,55.0,0.25,1
2024-01-01 08:00:00,11.631793,88.368207,100.0,55.0,0.25,1


In [5]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

def generate_jiat_dummy_data(days=30, interval_minutes=60):
    np.random.seed(42)
    
    # 1. Konstanta Spesifikasi Sumur & Pompa
    KEDALAMAN_SUMUR = 100.0
    KEDALAMAN_SENSOR = 55.0
    DIAMETER = 0.25 
    DEBIT_POMPA_LPS = 10.0 # Contoh Debit Spesifikasi Pompa: 10 Liter/Detik
    
    # Setup Index Waktu 
    periods = int((days * 24 * 60) / interval_minutes)
    time_index = pd.date_range(start='2026-01-01', periods=periods, freq=f'{interval_minutes}min')
    
    tma_list = []
    volume_discharge_list = [] # List untuk menampung jejak volume air yang ditarik 
    
    current_tma = 1.5 
    
    # 3. Simulasi Hidrologi Harian
    for dt in time_index:
        hour = dt.hour
        discharge_on = (6 <= hour < 9) or (15 <= hour < 18)
        
        # PENGHITUNGAN PERTUMBUHAN VOLUME BILA POMPA AKTIF
        if discharge_on:
            # Kalkulusi: Konversi Debit Pompa (L/s -> meter kubik per menit)
            # 10 Liter/Dtk = 600 Liter/Menit = 0.6 m3/Menit
            debit_m3_per_menit = (DEBIT_POMPA_LPS * 60) / 1000.0
            
            # Volume terambil dalam 1 interval array (misal 60 menit)
            volume_terambil = debit_m3_per_menit * interval_minutes
            
            target_tma = 28.0 
            current_tma += (target_tma - current_tma) * 0.25 
        else:
            volume_terambil = 0.0 # Saat recovery, debit pompa Off (0 m3)
            
            static_tma = 1.5
            current_tma += (static_tma - current_tma) * 0.12 
            
        noise = np.random.normal(0, 0.2)
        simulated_tma = current_tma + noise
        simulated_tma = max(0.0, min(30.0, simulated_tma))
        
        tma_list.append(simulated_tma)
        volume_discharge_list.append(volume_terambil) # Simpan record volume ke array
        
    df = pd.DataFrame({
        'Muka_Air_Tanah': tma_list,
        'Volume_Discharge_m3': volume_discharge_list # Feature Baru!
    }, index=time_index)
    
    df.index.name = 'Timestamp' 
    df['Data_Air_Tanah'] = KEDALAMAN_SUMUR - df['Muka_Air_Tanah']
    df['Kedalaman_Sumur'] = KEDALAMAN_SUMUR
    df['Kedalaman_Sensor'] = KEDALAMAN_SENSOR
    df['Diameter_Sumur'] = DIAMETER
    df['Discharge_Status'] = df.index.hour.map(lambda h: 1 if (6 <= h < 9) or (15 <= h < 18) else 0)
    
    return df

# ==========================================
# EKSEKUSI & PENGUJIAN VISUAL EDA AWAL
# ==========================================
df_sumur = generate_jiat_dummy_data(days=62, interval_minutes=60) 

print("Data Sample Teratas:")
display(df_sumur.head())

# VISUALISASI MENGGUNAKAN PLOTLY
fig = go.Figure()

# 1. Tambahkan Kurva Muka Air Tanah (TMA)
fig.add_trace(go.Scatter(
    x=df_sumur.index, 
    y=df_sumur['Muka_Air_Tanah'],
    mode='lines',
    name='Muka Air Tanah (TMA)',
    line=dict(color='#ef4444', width=2.5)
))

# 2. Tambahkan Kurva Data Air Tanah (Sisa)
fig.add_trace(go.Scatter(
    x=df_sumur.index, 
    y=df_sumur['Data_Air_Tanah'],
    mode='lines',
    name='Sisa Data Air Tanah',
    line=dict(color='#3b82f6', width=2.5)
))

# 3. Warnai Background pada Tiap Fase Discharge (Warna Pink)
discharge_data = df_sumur[df_sumur['Discharge_Status'] == 1]
for timestamp, row in discharge_data.iterrows():
    fig.add_vrect(
        x0=timestamp, 
        x1=timestamp + pd.Timedelta(minutes=60),
        fillcolor="green", 
        opacity=0.1,    
        layer="below",  
        line_width=0,
    )

# 4. Konfigurasi Kosmetik (Layout)
fig.update_layout(
    title='<b>Analisis Hubungan Sumur: Fluktuasi 2x Siklus Harian TMA vs Sisa Air</b>',
    xaxis_title='Waktu Simulasi',
    yaxis_title='Kedalaman / Ketinggian (Meter)',
    hovermode='x unified', 
    template='plotly_dark',
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.show()

Data Sample Teratas:


,Muka_Air_Tanah,Volume_Discharge_m3,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
Timestamp,,,,,,,
2026-01-01 00:00:00,1.599343,0.0,98.400657,100.0,55.0,0.25,0
2026-01-01 01:00:00,1.472347,0.0,98.527653,100.0,55.0,0.25,0
2026-01-01 02:00:00,1.629538,0.0,98.370462,100.0,55.0,0.25,0
2026-01-01 03:00:00,1.804606,0.0,98.195394,100.0,55.0,0.25,0
2026-01-01 04:00:00,1.453169,0.0,98.546831,100.0,55.0,0.25,0


In [9]:
import numpy as np
from scipy.stats import linregress

# Asumsi durasi pompa nyala: 15:00 - 18:00 (Durasi = 3 Jam / 180 Menit)
t_p_menit = 180.0 

# Kita tarik ulang regression parameter dari cell sebelumnya
df_theis = df_sumur.loc['2026-01-01 19:00:00':'2026-01-02 05:00:00'].copy()
t_start = pd.to_datetime('2026-01-01 15:00:00')
t_stop  = pd.to_datetime('2026-01-01 18:00:00')

df_theis['t_ratio'] = ((df_theis.index - t_start).total_seconds() / 60.0) / ((df_theis.index - t_stop).total_seconds() / 60.0)
df_theis['s_prime'] = df_theis['Muka_Air_Tanah'] - df_sumur.loc[t_start, 'Muka_Air_Tanah']

# Ekstrak persamaan regresi linear logaritmik
slope, intercept, _, _, _ = linregress(np.log10(df_theis['t_ratio']), df_theis['s_prime'])

# ========================================================
# RUMUS PENYELESAIAN WAKTU (t') THEIS
# ========================================================
def hitung_theis_t_prime(target_s_prime):
    # Hitung nilai pada Sumbu X Theis [ log10(t/t') ]
    log_val = (target_s_prime - intercept) / slope
    theis_ratio = 10 ** log_val 
    
    # Syarat Mutlak Fisika Theis: (t/t') HARUS dominan bernilai > 1.0 
    if theis_ratio <= 1.0:
        return "INFINITY (Pemulihan Mustahil: Akuifer mengalami Residual/Defisit Permanen)"
        
    waktu_t_prime_menit = t_p_menit / (theis_ratio - 1.0)
    return f"{waktu_t_prime_menit / 60.0:.2f} Jam" # Konversi Menit ke Jam

# --- UJI EKSPLORASI PREDIKSI THEIS ---
print("╔══════════════════════════════════════════════════════════════════╗")
print("║  PREDIKSI WAKTU RECOVERY (t') AKUIFER BERDASARKAN RUMUS THEIS    ║")
print("╚══════════════════════════════════════════════════════════════════╝")

# Momen ketika Air Sembuh 100% tanpa sisa drawdown sedikitpun
print(f"1. Prediksi waktu agar Pulih 100% (s' = 0.0m) : {hitung_theis_t_prime(0.0)}")

# Momen air sisa 3 Meter dari garis aman semula
print(f"2. Prediksi waktu agar air turun ke s' = 3.0m : {hitung_theis_t_prime(3.0)}")

# Momen air sisa 5 Meter (fase recovery paruh masa)
print(f"3. Prediksi waktu agar air turun ke s' = 5.0m : {hitung_theis_t_prime(5.0)}")


╔══════════════════════════════════════════════════════════════════╗
║  PREDIKSI WAKTU RECOVERY (t') AKUIFER BERDASARKAN RUMUS THEIS    ║
╚══════════════════════════════════════════════════════════════════╝
1. Prediksi waktu agar Pulih 100% (s' = 0.0m) : 1.88 Jam
2. Prediksi waktu agar air turun ke s' = 3.0m : 1.15 Jam
3. Prediksi waktu agar air turun ke s' = 5.0m : 0.86 Jam


In [7]:
# 1. Hitung laju perubahannya (+ naik memompa, - turun recovery)
df_sumur['Laju_TMA_Diff'] = df_sumur['Muka_Air_Tanah'].diff()

# 2. Tandai Fase Recovery (Apabila Muka air turun menuju landai / negatif)
df_sumur['Mode_Recovery'] = df_sumur['Laju_TMA_Diff'] < -0.05 

# 3. Pisahkan kluster ID per kejadian untuk satu siklus (biar event antar hari terpisah)
# Angka cumsum akan bertambah jika Mode_Recovery disela/diinterupsi Pompa Nyala (False)
df_sumur['Recovery_Event_ID'] = (~df_sumur['Mode_Recovery']).cumsum()

# 4. Deteksi otomatis resolusi waktu data Anda (Misal 60min diconvert menjadi 1.0 hr)
# (Teknik ini menjaga algoritma tetap dinamis walau parameter pemanggilan fungsi Anda ubah-ubah)
jam_per_baris = (df_sumur.index[1] - df_sumur.index[0]).total_seconds() / 3600.0

# 5. Filter & Kalkulasikan Total Waktunya
df_hanya_recovery = df_sumur[df_sumur['Mode_Recovery']]
tabel_rekap_recovery = df_hanya_recovery.groupby('Recovery_Event_ID').agg(
    Waktu_Mulai=('Laju_TMA_Diff', lambda baris: baris.index.min()),
    Waktu_Terakhir=('Laju_TMA_Diff', lambda baris: baris.index.max()),
    Jumlah_Data=('Laju_TMA_Diff', 'count')
).reset_index(drop=True)

# 6. HITUNG TOTAL WAKTU (Total Recovery Time dalam Hours)
tabel_rekap_recovery['Total_Recovery_Time (Hours)'] = tabel_rekap_recovery['Jumlah_Data'] * jam_per_baris

# Format & Munculkan Data Tabel Terbaru
print("Tabel Rekap Total Waktu Recovery Sumur (Hours):")
display(tabel_rekap_recovery[['Waktu_Mulai', 'Waktu_Terakhir', 'Total_Recovery_Time (Hours)']].head(10))

Tabel Rekap Total Waktu Recovery Sumur (Hours):


,Waktu_Mulai,Waktu_Terakhir,Total_Recovery_Time (Hours)
0,2026-01-01 01:00:00,2026-01-01 01:00:00,1.0
1,2026-01-01 04:00:00,2026-01-01 04:00:00,1.0
2,2026-01-01 09:00:00,2026-01-01 14:00:00,6.0
3,2026-01-01 18:00:00,2026-01-02 05:00:00,12.0
4,2026-01-02 09:00:00,2026-01-02 14:00:00,6.0
5,2026-01-02 18:00:00,2026-01-03 05:00:00,12.0
6,2026-01-03 09:00:00,2026-01-03 14:00:00,6.0
7,2026-01-03 18:00:00,2026-01-04 02:00:00,9.0
8,2026-01-04 04:00:00,2026-01-04 05:00:00,2.0
9,2026-01-04 09:00:00,2026-01-04 14:00:00,6.0


In [4]:
df_sumur[df_sumur["Discharge_Status"] == 1]

,Muka_Air_Tanah,Volume_Discharge_m3,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
Timestamp,,,,,,,
2026-01-01 06:00:00,8.440843,36.0,91.559157,100.0,55.0,0.25,1
2026-01-01 07:00:00,13.247237,36.0,86.752763,100.0,55.0,0.25,1
2026-01-01 08:00:00,16.726418,36.0,83.273582,100.0,55.0,0.25,1
2026-01-01 15:00:00,13.348654,36.0,86.651346,100.0,55.0,0.25,1
2026-01-01 16:00:00,16.893268,36.0,83.106732,100.0,55.0,0.25,1
...,...,...,...,...,...,...,...
2026-03-03 07:00:00,15.234854,36.0,84.765146,100.0,55.0,0.25,1
2026-03-03 08:00:00,18.490804,36.0,81.509196,100.0,55.0,0.25,1
2026-03-03 15:00:00,13.964793,36.0,86.035207,100.0,55.0,0.25,1


In [ ]:
df_sumur = df_sumur.reset_index()
# df_sumur.to_csv("data_dummy_tmat_12h.csv", index=False)
df_sumur

,Timestamp,Muka_Air_Tanah,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
0,2026-01-01 00:00:00,1.599343,98.400657,100.0,55.0,0.25,0
1,2026-01-01 01:00:00,1.472347,98.527653,100.0,55.0,0.25,0
2,2026-01-01 02:00:00,1.629538,98.370462,100.0,55.0,0.25,0
3,2026-01-01 03:00:00,1.804606,98.195394,100.0,55.0,0.25,0
4,2026-01-01 04:00:00,1.453169,98.546831,100.0,55.0,0.25,0
...,...,...,...,...,...,...,...
1483,2026-03-03 19:00:00,15.866048,84.133952,100.0,55.0,0.25,0
1484,2026-03-03 20:00:00,14.257395,85.742605,100.0,55.0,0.25,0
1485,2026-03-03 21:00:00,12.873481,87.126519,100.0,55.0,0.25,0
1486,2026-03-03 22:00:00,11.060855,88.939145,100.0,55.0,0.25,0


In [ ]:
import pandas as pd
import numpy as np
import math

def format_waktu(waktu_menit_total: float) -> str:
    """Mengubah total menit Theis menjadi format baca yang ramah (Hari, Jam, Menit)"""
    total_menit = int(round(waktu_menit_total))
    if total_menit < 60:
        return f"{total_menit} Menit"
        
    menit = total_menit % 60
    total_jam = total_menit // 60
    
    if total_jam < 24:
        if menit == 0:
            return f"{total_jam} Jam"
        return f"{total_jam} Jam {menit} Menit"
    else:
        hari = total_jam // 24
        jam = total_jam % 24
        res = f"{hari} Hari"
        if jam > 0:
            res += f" {jam} Jam"
        if menit > 0:
            res += f" {menit} Menit"
        return res

def estimate_theis_recovery(df, target_date, Q_m3_day=864, target_sisa_m=0.10):
    """
    Menghitung Estimasi Recovery theis untuk satu siklus harian.
    Referensi: Kruseman & De Ridder, 2000.
    - s' = (2.3 Q) / (4 pi T) * log10(t/t')
    """
    # 1. Ambil siklus data pada satu hari spesifik
    try:
        day_data = df.loc[target_date].copy()
    except KeyError:
        return f"Warning: Data untuk pencarian tanggal {target_date} tidak ada di CSV."
    
    # 2. Kalibrasi TMA Statis: Rata-rata muka air dini hari / sebelum Discharge
    static_tma = day_data.between_time('00:00', '05:00')['Muka_Air_Tanah'].mean()
    
    # 3. Durasi Discharge (tp = jam menyala)
    pump_on_data = day_data[day_data['Discharge_Status'] == 1]
    if len(pump_on_data) == 0:
        return "Tidak terdeteksi aktivitas Discharge pada hari ini."
    tp_hours = len(pump_on_data)
    
    # Cari timestamp tepat ketika mesin discharge dimatikan (Puncak Drawdown s_ma)
    waktu_mati = pump_on_data.index[-1]
    
    # 4. Metode 2 Titik (Two-Points method Theis Residual Drawdown) 
    # Mencari nilai t' = 1 (1 jam pasca mati) dan t' = 2 (2 jam pasca mati)
    t_prime_1_time = waktu_mati + pd.Timedelta(hours=1)
    t_prime_2_time = waktu_mati + pd.Timedelta(hours=2)
    
    if t_prime_1_time not in day_data.index or t_prime_2_time not in day_data.index:
        return "Data Kurang Panjang: Butuh minimal 2 baris jam setelah Discharge off."
        
    s_prime_1 = day_data.loc[t_prime_1_time, 'Muka_Air_Tanah'] - static_tma
    s_prime_2 = day_data.loc[t_prime_2_time, 'Muka_Air_Tanah'] - static_tma
    
    if s_prime_1 <= 0 or s_prime_2 <= 0:
        return "Anomali Theis: Nilai Air pemulihan melangkahi muka statis alam."
    if s_prime_2 >= s_prime_1:
         return "Anomali Fisik: Kurva recovery justru turun/tambah dalam."
         
    t_prime_1 = 1 # Jam
    t_prime_2 = 2 # Jam
    
    # 5. PERHITUNGAN KEMIRINGAN LOGARITMIK (SLOPE)
    ratio_1 = (tp_hours + t_prime_1) / t_prime_1
    ratio_2 = (tp_hours + t_prime_2) / t_prime_2
    
    log_diff = math.log10(ratio_1) - math.log10(ratio_2)
    delta_s = s_prime_1 - s_prime_2
    slope = delta_s / log_diff
    
    # 6. MENCARI NILAI TRANSMISIVITAS (T)
    T = (2.30 * Q_m3_day) / (4 * math.pi * slope)
    
    # 7. EKSTRAPOLASI WAKTU RECOVERY (Mencapai batas 10 cm dari muka statis awal)
    if s_prime_2 <= target_sisa_m:
         return {"Status": "Sudah stabil / Recovery Selesai", "Transmisivitas_m2_hari": round(T, 2), "Sisa Waktu": "0 Menit"}
         
    target_log_ratio = target_sisa_m / slope
    ratio_val = 10**target_log_ratio
    
    if ratio_val <= 1.01:
         sisa_jam = 0
    else:
         # Pembalikan Theis Interpolasi Waktu: t' = tp / (10^(s/slope) - 1)
         t_prime_target = tp_hours / (ratio_val - 1)
         sisa_jam = max(0, t_prime_target - t_prime_2)
         
    waktu_formatted = format_waktu(sisa_jam * 60)
    
    return {
        "Tanggal Analisis": target_date,
        "Kondisi Air": "Membutuhkan Waktu Recovery",
        "Muka Air Statis (Normal)": f"{round(static_tma, 3)} m",
        "Kedalaman Maks Drawdown": f"{round(day_data.loc[waktu_mati, 'Muka_Air_Tanah'], 3)} m",
        "Transmisivitas (T) Akuifer": f"{round(T, 2)} m²/hari",
        "Estimasi Estimasi Recovery Sisa (s' = 0.1m)": waktu_formatted
    }

# ==========================================
# TEST/ RUN LOGIC DENGAN CSV
# ==========================================
csv_path = r'D:\RnD_JIAT\note\data_tmat_dummy.csv'

# Baca Format File dan Jadikan Index Time

# df_main = pd.read_csv(csv_path)
df_main = df_sumur
df_main['Timestamp'] = pd.to_datetime(df_main['Timestamp'])
df_main.set_index('Timestamp', inplace=True)

# Mengeksekusi Perhitungan Rumus Theis untuk Tanggal 1 Januari (Terserah bisa ganti tanggal)
# Default menggunakan Q_m3_day = 864 yang merupakan representasi Fisika dari 10 Liter/detik.
hasil_theis = estimate_theis_recovery(df_main, target_date='2026-01-01')

print("=== Hasil Analisis Theis Recovery (CSV JIAT) ===")
# Pretty Print untuk menata Dictionary hasil return

for key, val in hasil_theis.items():
    print(f"- {key}: {val}")


=== Hasil Analisis Theis Recovery (CSV JIAT) ===
- Tanggal Analisis: 2026-01-01
- Kondisi Air: Membutuhkan Waktu Recovery
- Muka Air Statis (Normal): 1.569 m
- Kedalaman Maks Drawdown: 19.885 m
- Transmisivitas (T) Akuifer: 18.88 m²/hari
- Estimasi Estimasi Recovery Sisa (s' = 0.1m): 8 Hari 21 Jam 16 Menit


In [45]:
df_sumur

,Timestamp,Muka_Air_Tanah,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
0,2026-01-01 00:00:00,1.599343,98.400657,100.0,55.0,0.25,0
1,2026-01-01 01:00:00,1.472347,98.527653,100.0,55.0,0.25,0
2,2026-01-01 02:00:00,1.629538,98.370462,100.0,55.0,0.25,0
3,2026-01-01 03:00:00,1.804606,98.195394,100.0,55.0,0.25,0
4,2026-01-01 04:00:00,1.453169,98.546831,100.0,55.0,0.25,0
...,...,...,...,...,...,...,...
1483,2026-03-03 19:00:00,15.866048,84.133952,100.0,55.0,0.25,0
1484,2026-03-03 20:00:00,14.257395,85.742605,100.0,55.0,0.25,0
1485,2026-03-03 21:00:00,12.873481,87.126519,100.0,55.0,0.25,0
1486,2026-03-03 22:00:00,11.060855,88.939145,100.0,55.0,0.25,0


In [44]:
import math

def format_waktu(waktu_menit_total: float) -> str:
    total_menit = int(round(waktu_menit_total))
    if total_menit < 60:
         return f"{total_menit} Menit"
        
    menit = total_menit % 60
    total_jam = total_menit // 60
    
    if total_jam < 24:
         if menit == 0: return f"{total_jam} Jam"
         return f"{total_jam} Jam {menit} Menit"
    else:
         hari = total_jam // 24
         jam = total_jam % 24
         res = f"{hari} Hari"
         if jam > 0: res += f" {jam} Jam"
         if menit > 0: res += f" {menit} Menit"
         return res

def estimate_night_recovery(df, target_date, Q_m3_day=864, target_sisa_m=0.10):
    """
    Ekstrapolasi Theis Recovery setelah Discharge Sore (Siklus Ke-2) Hari Tersebut.
    """
    # Filter 1 Hari
    try:
        day_data = df.loc[str(target_date)].copy()
    except KeyError:
        return "Data tanggal tersebut tidak ada."
        
    # Muka statis absolut ada di waktu subuh (jam 00:00 - 05:00) karena sumur rehat sangat lama
    static_tma = day_data.between_time('00:00', '05:00')['Muka_Air_Tanah'].mean()
    
    # === ANALISIS FASE SORE ===
    # Waktu operasional sore adalah 3 jam (15:00 sampai menjelang jam 18:00)
    tp_hours = 3 
    
    # Timestamp tepat letikan pemompaan terakhir kali ON:
    waktu_mati = pd.to_datetime(f"{target_date} 17:00:00")
    if waktu_mati not in day_data.index:
         return "Siklus sore tidak ditemukan."
        
    # Titik Theis pasca mesin log-off: t'=1 (Jam 18:00), t'=2 (Jam 19:00)
    t_prime_1_time = waktu_mati + pd.Timedelta(hours=1)
    t_prime_2_time = waktu_mati + pd.Timedelta(hours=2)
    
    s_prime_1 = day_data.loc[t_prime_1_time, 'Muka_Air_Tanah'] - static_tma
    s_prime_2 = day_data.loc[t_prime_2_time, 'Muka_Air_Tanah'] - static_tma
    
    if s_prime_1 <= 0 or s_prime_2 <= 0:
         return "Air di akuifer sudah kembali menyentuh level statis."
        
    t_prime_1 = 1 # 1 Jam
    t_prime_2 = 2 # 2 Jam
    
    # --- RUMUS THEIS KRUSEMAN ---
    ratio_1 = (tp_hours + t_prime_1) / t_prime_1
    ratio_2 = (tp_hours + t_prime_2) / t_prime_2
    
    log_diff = math.log10(ratio_1) - math.log10(ratio_2)
    delta_s = s_prime_1 - s_prime_2
    
    if log_diff <= 0 or delta_s <= 0:
         return "Akurasi data terlalu rendah / anomali kurva pemulihan."
         
    slope = delta_s / log_diff
    T = (2.30 * Q_m3_day) / (4 * math.pi * slope) # Transmisivitas
    
    # Jika air kurang dari target sisa, dianggap sudah selesai recover (stabil)
    if s_prime_2 <= target_sisa_m:
         sisa_jam = 0
    else:
         target_log_ratio = target_sisa_m / slope
         ratio_val = 10**target_log_ratio
         if ratio_val <= 1.01:
             sisa_jam = 0
         else:
             # Rumus Pembalikan Ekstrapolasi: sisa waktu dari target_log
             t_prime_target = tp_hours / (ratio_val - 1)
             sisa_jam = max(0, t_prime_target - t_prime_2)
             
    waktu_formatted = format_waktu(sisa_jam * 60)
    
    return {
        "Siklus": f"Shutdown Sore, {target_date}",
        "Muka Air Awal Hari (Statis)": f"{round(static_tma, 2)} m",
        "Waktu Pompa Stop": str(waktu_mati.time()),
        "Penurunan Maksimal (Drawdown)": f"{round(day_data.loc[waktu_mati, 'Muka_Air_Tanah'], 2)} m",
        "Sisa Drawdown Jam ke-2 Paska Stop": f"{round(s_prime_2, 2)} m",
        "Kapasitas Porositas / Transmisivitas (T)": f"{round(T, 2)} m²/hari",
        "Estimasi Butuh Pemulihan Lengkap": waktu_formatted 
    }

# ==========================================
# CONTOH PENERAPAN
# ==========================================
# Ambil simulasi satu siklus raw sembarang hari (misal 5 Januari)
print("\n--- ESTIMASI THEIS RECOVERY ---")
hasil_prediksi = estimate_night_recovery(df_sumur, '2026-01-01')

if isinstance(hasil_prediksi, dict):
    for key, val in hasil_prediksi.items():
        print(f"{key:<45}: {val}")
else:
    print(hasil_prediksi)


--- ESTIMASI THEIS RECOVERY ---
Data tanggal tersebut tidak ada.


In [43]:
df_sumur

,Timestamp,Muka_Air_Tanah,Data_Air_Tanah,Kedalaman_Sumur,Kedalaman_Sensor,Diameter_Sumur,Discharge_Status
0,2026-01-01 00:00:00,1.599343,98.400657,100.0,55.0,0.25,0
1,2026-01-01 01:00:00,1.472347,98.527653,100.0,55.0,0.25,0
2,2026-01-01 02:00:00,1.629538,98.370462,100.0,55.0,0.25,0
3,2026-01-01 03:00:00,1.804606,98.195394,100.0,55.0,0.25,0
4,2026-01-01 04:00:00,1.453169,98.546831,100.0,55.0,0.25,0
...,...,...,...,...,...,...,...
1483,2026-03-03 19:00:00,15.866048,84.133952,100.0,55.0,0.25,0
1484,2026-03-03 20:00:00,14.257395,85.742605,100.0,55.0,0.25,0
1485,2026-03-03 21:00:00,12.873481,87.126519,100.0,55.0,0.25,0
1486,2026-03-03 22:00:00,11.060855,88.939145,100.0,55.0,0.25,0
